In [1]:
%pip install qutip
%pip install numpy

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


(a) Defining the gates

- Define the Pauli gates, and the Hadamard, S and T gates and the identity "gate" (will be useful later) as 2x2 matrices. We recommend to use numpy arrays.
- Define a function that builds a rotation gate rotation(ax, theta) for arbitrary rotation axis ax and angle theta. The function should return the rotation operator as a 2x2 matrix (numpy array). You may find the scipy function `expm()` useful for this. Test your implementation on some cases where you know what should come out (which one should generally do for every line of code one writes ;-).

In [2]:
import sys
import os

# Python path so notebook can find bell_state
sys.path.insert(0, os.path.abspath("src"))

from bell_state.operators import (
    I, X, Y, Z, H, S, T, P0, P1,
    controlled_gate, projectors, U_N_qubits, U_one_gate, U_two_gates,
    ket0, ket1, ket_plus, ket_minus, rotation_gate, toffoli_gate 
)

#### Visualization on the Bloch sphere

if we parameterize a state as $|\psi\rangle =\cos(\theta/2)|0\rangle+e^{i\varphi}\sin(\theta/2)|1\rangle$, where $\theta$ and $\varphi$ are the azimuthal and polar angle defining a point on the surface of the unit sphere in 3D, then the cartesian coordinates of that point correspond to the expectation values of the three Pauli operators.

We now want to visualize single-qubit states and explore the action of different gates in this way. For this, the QuTiP librabry provides the function Bloch(). QuTiP is a very powerful module for numerical simulation of quantum problems (install and explore qutip it you haven't done so yet!).

Use the code below, where a single spin is represented on the Bloch sphere, to familiarize yourself with the Bloch class. (This will only work if the function rotation() is defined and behaves as described in problem 1a). Make sure you understand what the `@ operator` does (dot-product, only available in python 3).

Now iteratively apply the gate THTH to convince yourself that it indeed is a rotation about an axis $\mathbf{n}^*=(\cos(\pi/8),\sin(\pi/8),\cos(\pi/8))$ (unnormalized), as you have shown on problem set 1 and that the rotation angles densely cover the full circle upon interative application. For this, generate a plot that contains all the states generated from $|0\rangle$ upon apply the gate 100 times.

*Optional:*

The action of gates can also be visualized by letting it act on a representative ensemble of states and collectively examine its
effect on them.
Generate a set of sample states, e.g. regularly spaced points on the unit circle in the x-z-plane and apply some of the gates we got to know on Problem Set 1 (H, T, HTH, THTH...) to all of these states. Plot all the points before and after applying the gates jointly on the Bloch sphere to see that what is happening is indeed a rotation about a certain axis. Note: the function `add_points()` of the Bloch class also accepts a list of points. Other potentially useful numpy functions are `linspace()` and `append()`.

In [ ]:
import qutip
import numpy as np

U = T @ H @ T @ H
initial_state = ket0()
final_states = [qutip.Qobj(initial_state)]
for _ in range(100):
    initial_state = U @ initial_state
    final_states.append(qutip.Qobj(initial_state))

b = qutip.Bloch(figsize=(3,3))
b.add_states(final_states)
b.show()

Multi-qubit gates and Bell_state

Consider a register of two qubits, which take the role of control and target. Show that the CNOT gate can be written as $CNOT = P_0\otimes I + P_1 \otimes X$, where $P_k=|k\rangle\langle k|$ is the projector on the single qubit state $|k\rangle$ and $\otimes$ denotes the tensor product, or Kronecker product (compute the matrix representation of CNOT from this expression).
This way of writing the CNOT gate is instructive as it explicitly distinguishes between the two states of the control qubit. The projectors onto the control qubit states can be read as: If the control qubit is in state $|0\rangle$, do nothing to the taget qubit (apply the identity). If the control qubit is in state $|1\rangle$, flip the taget qubit (apply the X gate). This formulation will also be helpful when we generalize to n-qubit registers. It allows us to implement any controlled unitary gate straight forwardly.

As a first application, we want to execute a quantum circuit that prepares the Bell state $(|00\rangle+|11\rangle)/\sqrt{2}$ (discussed in the lecture):
Initialize the register in state $|00\rangle$, apply a H gate to the first qubit, apply a CNOT gate. Verify by hand that this prepares a Bell state. Implement the preparation numerically: Sequentially apply the gates to the state using the `@` operator. Use the numpy function `kron()` to implement a Hadamard gate on the first qubit (i.e. build the matrix representation of the operator $H\otimes I$) and the CNOT gate (as done above) in this register.

In [4]:
import numpy as np
CNOT = controlled_gate(X, 0, 1, N=2)  
print(CNOT)

[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]]


In [5]:
import numpy as np
CNOT = controlled_gate(X, 1, 0, N=2)  
print(CNOT)

[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]]


In [26]:
import numpy as np
CZ = controlled_gate(Z, 0, 1, N=2)  
print("CZ:\n", CZ)

CZ:
 [[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j -1.+0.j]]


In [27]:
import numpy as np
CZ = controlled_gate(Z, 1, 0, N=2)  
print("CZ:\n", CZ)

CZ:
 [[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j -1.+0.j]]


In [6]:
import numpy as np

encoded_state = np.kron(ket0(), ket0()) 
H_op = np.kron(H, I)
state_after_H = H_op @ encoded_state
CNOT = controlled_gate(X, 0, 1, N=2)  
phi_plus_state = CNOT @ state_after_H 
print("Initial state:", encoded_state)
print()
print("Φ⁺ (Phi Plus):", phi_plus_state)

Initial state: [1.+0.j 0.+0.j 0.+0.j 0.+0.j]

Φ⁺ (Phi Plus): [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


In [7]:
import numpy as np

encoded_state = np.kron(ket1(), ket0()) 
H_op = np.kron(H, I)
state_after_H = H_op @ encoded_state
CNOT = controlled_gate(X, 0, 1, N=2)  
phi_minus_state = CNOT @ state_after_H
print("Initial state:", encoded_state)
print()
print("Φ⁻ (Phi minus):", phi_minus_state)

Initial state: [0.+0.j 0.+0.j 1.+0.j 0.+0.j]

Φ⁻ (Phi minus): [ 0.70710678+0.j  0.        +0.j  0.        +0.j -0.70710678+0.j]


In [8]:
import numpy as np

encoded_state = np.kron(ket0(), ket1()) 
H_op = np.kron(H, I)
state_after_H = H_op @ encoded_state
CNOT = controlled_gate(X, 0, 1, N=2)  
Psi_Plus_state = CNOT @ state_after_H 
print("Initial state:", encoded_state)
print()
print("Ψ⁺ (Psi Plus):", Psi_Plus_state)

Initial state: [0.+0.j 1.+0.j 0.+0.j 0.+0.j]

Ψ⁺ (Psi Plus): [0.        +0.j 0.70710678+0.j 0.70710678+0.j 0.        +0.j]


In [9]:
import numpy as np

encoded_state = np.kron(ket1(), ket1()) 
H_op = np.kron(H, I)
state_after_H = H_op @ encoded_state
CNOT = controlled_gate(X, 0, 1, N=2)  
Psi_minus_state = CNOT @ state_after_H 
print("Initial state:", encoded_state)
print()
print("Ψ⁻ (Psi Plus):", Psi_minus_state)

Initial state: [0.+0.j 0.+0.j 0.+0.j 1.+0.j]

Ψ⁻ (Psi Plus): [ 0.        +0.j  0.70710678+0.j -0.70710678+0.j  0.        +0.j]


# GHZ State — Theory Note
The **GHZ (Greenberger–Horne–Zeilinger) state** is a multi-qubit entangled quantum state.  
For three qubits, the GHZ state is:

\[
|GHZ\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)
\]

This state represents **maximal entanglement** among three qubits.

In [ ]:
import numpy as np 

encoded_state = np.kron(ket0(), np.kron (ket0(), ket0()))
H_q0 = np.kron(H, np.kron(I, I))
state_after_H =  H_q0@ encoded_state
CNOT0_2 = controlled_gate(X, 0, 2, 3) 
state_after_CNOT1_2=CNOT0_2@state_after_H
CNOT0_1 = controlled_gate(X, 0, 1, 3)
state_after_CNOT0_1=CNOT0_1@state_after_CNOT1_2
CNOT1_0 = controlled_gate(X, 1, 0, 3)
GHZ= CNOT1_0@state_after_CNOT0_1
print(GHZ)

In [ ]:
import numpy as np

S01 = P0 @ X @ P1   
S10 = P1 @ X @ P0   

SWAP = (np.kron(P0, P0) +
        np.kron(P1, P1) +
        np.kron(S01, S10) +
        np.kron(S10, S01))

print(SWAP)

In [24]:
import numpy as np

theta = np.pi       
n = (1, 0, 0)       
R = rotation_gate(theta, n)
CC_R= toffoli_gate(R)
print("CC_R:\n", CC_R)

CC_R:
 [[1.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 1.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 1.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 1.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  1.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 1.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 6.123234e-17+0.j 0.000000e+00-1.j]
 [0.00

In [23]:
import numpy as np

theta = np.pi
n = (0, 1, 0) 
R = rotation_gate(theta, n)
CC_R= toffoli_gate(R)
print("CC_R:\n", CC_R)

CC_R:
 [[ 1.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j
   0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j]
 [ 0.000000e+00+0.j  1.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j
   0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j]
 [ 0.000000e+00+0.j  0.000000e+00+0.j  1.000000e+00+0.j  0.000000e+00+0.j
   0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j]
 [ 0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  1.000000e+00+0.j
   0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j]
 [ 0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j
   1.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j]
 [ 0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j
   0.000000e+00+0.j  1.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j]
 [ 0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j  0.000000e+00+0.j
   0.000000e+00+0.j  0.00

In [28]:
import numpy as np

theta = np.pi
n = (0, 0,1) 
R = rotation_gate(theta, n)
CC_R= toffoli_gate(R)
print("CC_R:\n", CC_R)

CC_R:
 [[1.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 1.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 1.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 1.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  1.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 1.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j]
 [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j
  0.000000e+00+0.j 0.000000e+00+0.j 6.123234e-17-1.j 0.000000e+00+0.j]
 [0.00

In [15]:
import numpy as np

theta= np.pi
n= (1/np.sqrt(2), 0, 1/np.sqrt(2))  
R = rotation_gate(theta, n)
CC_R= toffoli_gate(R)
print("CC_R:\n", CC_R)

CC_R:
 [[1.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j        ]
 [0.000000e+00+0.j         1.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j        ]
 [0.000000e+00+0.j         0.000000e+00+0.j
  1.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j        ]
 [0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         1.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j        ]
 [0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j
  1.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j         0.000000e+00+0.j        ]
 [0.000000e+00+0.j         0.000000e+00+0.j
  0.000000e+00+0.j      

In [16]:
CC_X = toffoli_gate(X)
print("CC_X:\n", CC_X)

CC_X:
 [[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j]]


In [18]:
CC_Y = toffoli_gate(Y)
print("CC_Y:\n", CC_Y)

CC_Y:
 [[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.-1.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+1.j 0.+0.j]]


In [19]:
CC_Z = toffoli_gate(Z)
print("CC_Z:\n", CC_Z)

CC_Z:
 [[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j -1.+0.j]]


In [20]:
CC_H= toffoli_gate(H)
print("CCX gate matrix (8x8):\n", CC_H)

CCX gate matrix (8x8):
 [[ 1.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j  1.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j  0.        +0.j  1.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j  1.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   1.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  1.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.70710678+0.j  0.70710678+0.j]
 [ 0.        +0.j  0.        +0.j  0.        

In [21]:
CC_T = toffoli_gate(T)
print("CC_T:\n", CC_T)

CC_T:
 [[1.        +0.j         0.        +0.j         0.        +0.j
  0.        +0.j         0.        +0.j         0.        +0.j
  0.        +0.j         0.        +0.j        ]
 [0.        +0.j         1.        +0.j         0.        +0.j
  0.        +0.j         0.        +0.j         0.        +0.j
  0.        +0.j         0.        +0.j        ]
 [0.        +0.j         0.        +0.j         1.        +0.j
  0.        +0.j         0.        +0.j         0.        +0.j
  0.        +0.j         0.        +0.j        ]
 [0.        +0.j         0.        +0.j         0.        +0.j
  1.        +0.j         0.        +0.j         0.        +0.j
  0.        +0.j         0.        +0.j        ]
 [0.        +0.j         0.        +0.j         0.        +0.j
  0.        +0.j         1.        +0.j         0.        +0.j
  0.        +0.j         0.        +0.j        ]
 [0.        +0.j         0.        +0.j         0.        +0.j
  0.        +0.j         0.        +0.j         1.     

In [22]:
CC_S = toffoli_gate(S)
print("CC_S:\n", CC_S)

CC_S:
 [[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+1.j]]
